# Урок 08 - Паттерн проектирования многоагентных систем


## Настройка


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Почему многоагентные системы?

Реальные задачи, такие как планирование поездок, требуют разных видов экспертизы — логистика, местные знания, бюджетирование и многое другое. Один агент, пытающийся справиться со всем, быстро становится негибким.

Многоагентные системы решают это через **специализацию**: каждый агент сосредотачивается на одной области экспертизы, что даёт результаты высокого качества по сравнению с универсалом. Они также улучшают **масштабируемость** — можно добавлять новых агентов (например, специалист по авиаперелётам, критик ресторанов) без переписывания существующего рабочего процесса. Агенты взаимодействуют через структурированный конвейер, передавая контекст от одного к другому.


## Создание специализированных агентов


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Создание последовательного рабочего процесса

`WorkflowBuilder` позволяет связывать агентов в ориентированный граф. Здесь мы создаем простой двухэтапный конвейер: **TravelPlanner** составляет маршрут, затем **TravelConcierge** проверяет и улучшает его.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Добавление дополнительных агентов в рабочий процесс

Одно из главных преимуществ паттерна с несколькими агентами — его простота расширения. Ниже мы добавляем агента **BudgetReviewer**, который проверяет план на соответствие бюджету путешественника, отмечает позиции, которые могут увеличить расходы сверх лимита, и предлагает более экономичные альтернативы. Теперь рабочий процесс запускает три агента последовательно:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Резюме

В этом уроке вы научились:

1. **Создавать специализированных агентов** — каждый с определенной ролью (планирование, консьерж, проверка бюджета).
2. **Связывать агентов в последовательный рабочий процесс** с помощью `WorkflowBuilder` и `add_edge`.
3. **Потоково выводить данные** из мультиагентного конвейера, отслеживая, какой агент говорит.
4. **Расширять рабочий процесс**, добавляя новых агентов в цепочку без изменения существующих.

Шаблон проектирования с несколькими агентами сохраняет каждого агента простым и при этом обеспечивает более содержательные и тщательно проверенные результаты, чем мог бы добиться любой отдельный агент.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:  
Этот документ был переведен с помощью автоматического сервиса перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, пожалуйста, учитывайте, что автоматические переводы могут содержать ошибки или неточности. Оригинальный документ на его родном языке следует считать авторитетным источником. Для важной информации рекомендуется использовать профессиональный перевод, выполненный человеком. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования данного перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
